# CHW Copilot — End-to-End Pipeline on Kaggle

**Competition:** Google AI Assistants for Data Tasks with Gemma

This notebook runs the full CHW Copilot pipeline on Kaggle GPU using **MedGemma only**:
1. Load MedGemma-4b-it for extraction, syndrome tagging, checklist, and SITREP
2. Extract structured encounters from typed CHW notes
3. Evaluate extraction quality
4. Run surveillance pipeline

**Runtime:** Kaggle T4 GPU (16GB VRAM)

---

## 0. Setup & Dependencies

In [19]:
# Install dependencies
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.40", "accelerate>=0.27", "jsonschema>=4.17",
    "pandas>=2.0", "torch"])
print("Dependencies installed ✅")

Dependencies installed ✅


In [21]:
import os, json, time, warnings
from pathlib import Path
import pandas as pd
import torch
warnings.filterwarnings("ignore")

# Detect environment
IS_KAGGLE = os.path.exists("/kaggle/working")
if IS_KAGGLE:
    ROOT = Path("/kaggle/input/datasets/kheziantomoa/support-files")
    OUT_DIR = Path("/kaggle/working")
else:
    ROOT = Path(".")
    OUT_DIR = Path(".")

print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"Root (input): {ROOT}")
print(f"Output dir: {OUT_DIR}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Environment: Kaggle
Root (input): /kaggle/input/datasets/kheziantomoa/support-files
Output dir: /kaggle/working
GPU available: True
GPU: Tesla T4


## 1. Load Prompts and Schemas

In [22]:
# Load prompt templates (Hardcoded to ensure latest version)
print("Loading hardcoded prompts...")

extraction_prompt = r"""You are a medical information extractor for Community Health Worker (CHW) field notes.

TASK: Extract structured clinical information from the CHW note below. Return ONLY valid JSON matching the schema.

RULES:
1. For each symptom, set value to "yes", "no", or "unknown".
2. If value is "yes", you MUST include an evidence_quote that is a VERBATIM, CHARACTER-FOR-CHARACTER substring copied from the note. Do NOT fix typos. Do NOT change numbers (e.g. "3" vs "three"). Do NOT paraphrase. If you cannot copy exactly, do not extract it.
3. If value is "unknown" or "no", evidence_quote MUST be null.
4. For each symptom with value "yes", also extract duration if stated (e.g., "3 days", "1 week", "since yesterday"). Set to null if not stated.
5. For onset_days: extract number of days if stated (e.g., "3 days" → 3), otherwise null.
6. For severity: only use "mild", "moderate", or "severe" if clearly stated or strongly implied. Otherwise "unknown".
7. For patient.age_group: use "infant" (<1yr), "child" (1-9yr), "adolescent" (10-17yr), "adult" (18-64yr), "elderly" (65+yr), or "unknown".
8. For patient.age_years: extract exact age if stated, otherwise null.
9. For patient.patient_id: extract if stated, otherwise "unknown".
10. For patient.pregnancy_status: only include if pregnancy is mentioned. Same {value, evidence_quote} rules.
11. For red_flags: only include if the note contains clear danger signs. Each must have evidence_quote.
12. Do NOT infer or hallucinate. If it's not in the note, mark as "unknown" or null.
13. Use syndromic language only — never diagnostic terms.

CORE SYMPTOMS TO EXTRACT (required):
- fever
- cough
- watery_diarrhea
- bloody_diarrhea
- vomiting
- rash
- difficulty_breathing

However if other symptoms not in the list are mentioned, include them in other_symptoms using snake_case keys (e.g. headache, sore_throat, abdominal_pain, muscle_aches, night_sweats, joint_pain, weight_loss, fatigue). Same {value, evidence_quote, duration} rules apply.

RED FLAGS (only if present):
- dehydration_signs (sunken eyes, dry mouth, skin pinch slow, no tears)
- unable_to_drink
- persistent_vomiting
- blood_in_stool
- high_fever (≥39°C or described as very high/burning)
- convulsions
- altered_consciousness (confused, unconscious, very sleepy)
- severe_malnutrition
- chest_indrawing

ENCOUNTER METADATA (extract if stated, otherwise defaults):
- chw_id: CHW identifier/name → string, default "unknown"
- visit_date: ISO date "YYYY-MM-DD" if stated → string|null
- visit_datetime: ISO datetime if stated → string|null
- encounter_sequence: order within worker's day → integer|null
- area_id: village/ward/neighbourhood → string, default "unknown"
- household_id: household/compound ID → string, default "unknown"
- gps: {lat, lon, accuracy_m} if coordinates stated → object|null

ACTIONS & OUTCOMES (extract if stated):
- treatments_given: array of meds/ORS/fluids mentioned → empty array if none
- referral: {value: "yes"|"no"|"unknown", destination: string, evidence_quote} → null if not stated
- follow_up: {value: "yes"|"no"|"unknown", follow_up_date: string|null, evidence_quote} → null if not stated

CHW NOTE:
{note_text}

RESPONSE FORMAT (Strict JSON):
{
  "symptoms": {
    "fever": { "value": "yes", "evidence_quote": "exact quote from note", "duration": "3 days" },
    "cough": { "value": "no", "evidence_quote": null, "duration": null },
    "watery_diarrhea": { "value": "unknown", "evidence_quote": null, "duration": null },
    ... (include all core symptoms)
  },
  "other_symptoms": {
    "headache": { "value": "yes", "evidence_quote": "complains of headache", "duration": "since morning" }
  },
  "red_flags": [
    { "flag": "danger_sign_name", "evidence_quote": "exact quote" }
  ],
  "patient": {
    "age_group": "child",
    "age_years": 3,
    "sex": "female",
    "patient_id": "unknown",
    "pregnancy_status": { "value": "no", "evidence_quote": null }
  },
  "encounter_metadata": {
    "chw_id": "unknown",
    "visit_date": "YYYY-MM-DD",
    "visit_datetime": null,
    "encounter_sequence": null,
    "area_id": "unknown",
    "household_id": "unknown",
    "gps": null
  },
  "actions_outcomes": {
    "treatments_given": ["ORS", "Zinc"],
    "referral": { "value": "yes", "destination": "health center", "evidence_quote": "referred to HC" },
    "follow_up": { "value": "yes", "follow_up_date": "2024-03-10", "evidence_quote": "visit tomorrow" }
  }
}

Return ONLY the JSON object. No explanation, no markdown fences.
"""

checklist_prompt = r"""You are a clinical completeness checker for Community Health Worker encounters.

TASK: Given the extracted encounter data and the original CHW note, identify the most important MISSING information that the CHW should still collect from the patient. Return a JSON checklist.

RULES:
1. Only ask about fields that are "unknown" or null in the encounter.
2. Maximum 5 questions, prioritized by clinical importance.
3. Each question must map to a specific encounter field.
4. Priority: "high" for danger-sign-related gaps, "medium" for syndrome-relevant gaps, "low" for demographics/metadata.
5. Questions must be simple, plain language a CHW can ask directly.
6. Do NOT suggest diagnostic tests or referrals — only information-gathering questions.
7. Consider ALL clinically relevant fields.
8. START WITH DEMOGRAPHIC CHECK: 
   - If patient is MALE, CHILD (<10), or ELDERLY (>50), do NOT ask about pregnancy.
   - If patient is > 2 years, do NOT ask about breastfeeding.
   - If patient is < 5 years, prioritize danger signs (unable to drink, vomiting everything).
9. Do not annoy the CHW with obvious questions if context implies they are irrelevant.

PRIORITY ORDER:
- High: fever status, dehydration signs, breathing difficulty, ability to eat/drink, pregnancy in women of reproductive age
- Medium: onset duration, vomiting, diarrhea character, rash, other symptoms mentioned vaguely
- Low: exact age, sex if unclear, household_id, area_id, CHW metadata

EXTRACTED ENCOUNTER:
{encounter_json}

ORIGINAL NOTE:
{note_text}

Return ONLY the JSON object with "encounter_id" and "questions" array. Each question has: "field" (dot notation like "symptoms.fever" or "other_symptoms.headache" or "patient.age_years"), "question" (plain language), "priority" ("high"|"medium"|"low"). No explanation.
"""

def load_prompt(name):
    # Fallback for others
    path = ROOT / "prompts" / f"{name}.txt"
    if path.exists():
        return path.read_text(encoding="utf-8")
    return ""

syndrome_prompt = load_prompt("syndrome_tagger")
sitrep_prompt = load_prompt("monitoring_sitrep")

print(f"Extraction prompt: {len(extraction_prompt)} chars")
print(f"Syndrome prompt: {len(syndrome_prompt)} chars")
print(f"Checklist prompt: {len(checklist_prompt)} chars")
print(f"SITREP prompt: {len(sitrep_prompt)} chars")
print("All prompts loaded ✅")


Extraction prompt: 3072 chars
Syndrome prompt: 1386 chars
Checklist prompt: 1567 chars
SITREP prompt: 1482 chars
All prompts loaded ✅


In [23]:
# Load schemas for validation
import jsonschema

def load_schema(name):
    path = ROOT / "schemas" / f"{name}.schema.json"
    with open(path, encoding="utf-8") as f:
        return json.load(f)

encounter_schema = load_schema("encounter")
syndrome_schema = load_schema("syndrome")
checklist_schema = load_schema("checklist")
sitrep_schema = load_schema("sitrep")
print("All schemas loaded ✅")

All schemas loaded ✅


## 2. Load MedGemma

**MedGemma-4b-it** (Google) — handles everything:
- Structured extraction from typed CHW notes
- Syndrome tagging
- Checklist generation
- SITREP generation

> **Note:** MedGemma is a gated model. You need to:
> 1. Accept the license at https://huggingface.co/google/medgemma-4b-it
> 2. Add your HuggingFace token as a Kaggle Secret named `HF_TOKEN`

In [24]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- HuggingFace authentication (for gated models like MedGemma) ---
HF_TOKEN = None
if IS_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    try:
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        print("HF_TOKEN loaded from Kaggle Secrets ✅")
    except Exception:
        print("⚠️  HF_TOKEN not found in Kaggle Secrets — MedGemma may fail to load")
else:
    HF_TOKEN = os.getenv("HF_TOKEN")
    if HF_TOKEN:
        print("HF_TOKEN loaded from environment ✅")
    else:
        print("⚠️  No HF_TOKEN set — MedGemma may fail to load")

HF_TOKEN loaded from Kaggle Secrets ✅


In [25]:
# Load MedGemma
print("Loading MedGemma-4b-it...")
t0 = time.time()

MEDGEMMA_ID = "google/medgemma-4b-it"
mg_tokenizer = AutoTokenizer.from_pretrained(
    MEDGEMMA_ID, trust_remote_code=True, token=HF_TOKEN
)
mg_model = AutoModelForCausalLM.from_pretrained(
    MEDGEMMA_ID,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    token=HF_TOKEN,
)
mg_model.eval()
device = next(mg_model.parameters()).device
print(f"MedGemma loaded on {device} in {time.time()-t0:.1f}s ✅")

Loading MedGemma-4b-it...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

MedGemma loaded on cuda:0 in 6.3s ✅


## 3. Helper Functions

In [26]:
import re

def parse_json_response(text):
    """Extract JSON from model response, handling code fences and extra text."""
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    m = re.search(r'```(?:json)?\s*\n(.*?)\n```', text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except json.JSONDecodeError:
            pass
    m = re.search(r'\{.*\}', text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except json.JSONDecodeError:
            pass
    return None


def run_medgemma(prompt, max_new_tokens=512):
    """Run MedGemma with chat template."""
    messages = [{"role": "user", "content": prompt}]
    input_text = mg_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = mg_tokenizer(input_text, return_tensors="pt").to(mg_model.device)
    with torch.no_grad():
        outputs = mg_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=mg_tokenizer.eos_token_id,
        )
    return mg_tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def enforce_evidence(encounter, note_text):
    """Downgrade 'yes' claims without valid evidence quotes."""
    note = (note_text or "").lower()
    downgrades = []
    for k, v in list(encounter.get("symptoms", {}).items()):
        if v.get("value") == "yes":
            q = v.get("evidence_quote")
            if not q or q.lower() not in note:
                encounter["symptoms"][k] = {"value": "unknown", "evidence_quote": None}
                downgrades.append(f"symptoms.{k}")
    other_symp = encounter.get("other_symptoms", {})
    if isinstance(other_symp, dict):
        for k, v in list(other_symp.items()):
            if isinstance(v, dict) and v.get("value") == "yes":
                q = v.get("evidence_quote")
                if not q or q.lower() not in note:
                    encounter["other_symptoms"][k] = {"value": "unknown", "evidence_quote": None}
                    downgrades.append(f"other_symptoms.{k}")
    valid_flags = []
    for flag in encounter.get("red_flags", []):
        if isinstance(flag, str):
            # Model returned plain string — drop it (no evidence)
            downgrades.append(f"red_flag:{flag}")
            continue
        q = flag.get("evidence_quote", "")
        if q and q.lower() in note:
            valid_flags.append(flag)
        else:
            downgrades.append(f"red_flag:{flag.get('flag','?')}")
    encounter["red_flags"] = valid_flags
    return encounter, downgrades

## 4. Quick Smoke Test (1 Note)

In [27]:
test_note = "Child 3yo male fever 3 days cough bad difficulty breathing rash on chest no diarrhea mother says not eating gave ORS referred health center"

print("=== MedGemma Extraction ===")
t0 = time.time()
extract_prompt = extraction_prompt.replace("{note_text}", test_note)
raw_extract = run_medgemma(extract_prompt, max_new_tokens=1024)
print(f"MedGemma extraction time: {time.time()-t0:.1f}s")
parsed_extract = parse_json_response(raw_extract)
if parsed_extract:
    print(json.dumps(parsed_extract, indent=2)[:800])
else:
    print("⚠️ Failed to parse extraction output:")
    print(raw_extract[:500])

=== MedGemma Extraction ===
MedGemma extraction time: 36.7s
{
  "fever": "yes",
  "evidence_quote": "difficulty breathing",
  "cough": "yes",
  "watery_diarrhea": "no",
  "bloody_diarrhea": "no",
  "vomiting": "unknown",
  "rash": "yes",
  "difficulty_breathing": "yes",
  "other_symptoms": {},
  "red_flags": [],
  "patient": {
    "age_group": "child",
    "age_years": "3",
    "patient_id": "unknown",
    "pregnancy_status": "unknown"
  },
  "encounter_metadata": {
    "chw_id": "unknown",
    "visit_date": null,
    "visit_datetime": null,
    "encounter_sequence": null,
    "area_id": "unknown",
    "household_id": "unknown",
    "gps": null
  },
  "actions_outcomes": {
    "treatments_given": [
      "ORS"
    ],
    "referral": {
      "value": "yes",
      "destination": "health center",
      "evidence_quote": "referred health center"
    },


In [28]:
print("=== MedGemma Syndrome Tagging ===")
tag_prompt = syndrome_prompt.replace("{encounter_json}", json.dumps(parsed_extract or {}, indent=2))
tag_prompt = tag_prompt.replace("{note_text}", test_note)
t0 = time.time()
raw_tag = run_medgemma(tag_prompt)
print(f"MedGemma time: {time.time()-t0:.1f}s")
parsed_tag = parse_json_response(raw_tag)
if parsed_tag:
    print(json.dumps(parsed_tag, indent=2)[:500])
else:
    print("⚠️ Failed to parse:")
    print(raw_tag[:500])

=== MedGemma Syndrome Tagging ===
MedGemma time: 13.4s
{
  "encounter_id": "unknown",
  "syndrome_tag": "respiratory_fever",
  "confidence": "high",
  "trigger_quotes": [
    "fever",
    "cough",
    "difficulty breathing",
    "rash",
    "difficulty breathing"
  ],
  "reasoning": "The note mentions fever, cough, and difficulty breathing, which meet the criteria for respiratory_fever. The presence of a rash is also relevant. The child is also 3 years old, which is a common age for respiratory infections."
}


In [29]:
print("=== MedGemma Checklist ===")
cl_prompt = checklist_prompt.replace("{encounter_json}", json.dumps(parsed_extract or {}, indent=2))
cl_prompt = cl_prompt.replace("{note_text}", test_note)
t0 = time.time()
raw_cl = run_medgemma(cl_prompt)
print(f"MedGemma time: {time.time()-t0:.1f}s")
parsed_cl = parse_json_response(raw_cl)
if parsed_cl:
    print(json.dumps(parsed_cl, indent=2)[:500])
else:
    print("⚠️ Failed to parse:")
    print(raw_cl[:500])

=== MedGemma Checklist ===
MedGemma time: 22.5s
{
  "encounter_id": "unknown",
  "questions": [
    {
      "field": "symptoms.vomiting",
      "question": "Has the child vomited?",
      "priority": "medium"
    },
    {
      "field": "symptoms.onset_days",
      "question": "How long has the child had a fever?",
      "priority": "medium"
    },
    {
      "field": "patient.age_years",
      "question": "What is the child's age in years?",
      "priority": "low"
    },
    {
      "field": "patient.pregnancy_status",
      "question": "I


## 5. Full Pipeline Function

In [33]:
def process_note(note_text, encounter_id, location_id, week_id):
    """Full pipeline: extract → enforce evidence → tag → checklist."""
    result = {"encounter_id": encounter_id, "errors": []}
    t0 = time.time()

    # Step 1: Extract with MedGemma
    try:
        ext_prompt = extraction_prompt.replace("{note_text}", note_text)
        raw = run_medgemma(ext_prompt, max_new_tokens=1024)
        parsed = parse_json_response(raw)
        if parsed is None:
            result["errors"].append("extraction_parse_fail")
            parsed = {}
    except Exception as e:
        result["errors"].append(f"extraction_error: {e}")
        parsed = {}

    # Normalize symptoms
    symptoms = parsed.get("symptoms", {})
    for key in ["fever","cough","watery_diarrhea","bloody_diarrhea","vomiting","rash","difficulty_breathing"]:
        claim = symptoms.get(key, {})
        if not isinstance(claim, dict):
            claim = {}
        val = str(claim.get("value", "unknown")).lower().strip()
        if val not in ("yes", "no"):
            val = "unknown"
        quote = claim.get("evidence_quote")
        if val != "yes":
            quote = None
        elif not (quote and isinstance(quote, str) and quote.strip()):
            quote = None
            val = "unknown"
        symptoms[key] = {"value": val, "evidence_quote": quote, "duration": claim.get("duration") if val == "yes" else None}

    # Normalize other_symptoms (FIXED)
    other_symp = {}
    raw_other = parsed.get("other_symptoms", {})
    if isinstance(raw_other, dict):
        for k, v in raw_other.items():
            # --- FIX: Ensure v is a dict before using .get() ---
            if not isinstance(v, dict):
                continue
            # -----------------------------------------------------
            val = str(v.get("value","unknown")).lower().strip()
            if val not in ("yes","no"): val = "unknown"
            q = v.get("evidence_quote")
            if val != "yes": q = None; dur = None
            elif not (q and isinstance(q, str) and q.strip()): q = None; val = "unknown"; dur = None
            else: dur = v.get("duration")
            other_symp[k] = {"value": val, "evidence_quote": q, "duration": dur}

    # Normalize patient
    pat = parsed.get("patient", {})
    if not isinstance(pat, dict): pat = {}
    age_group = str(pat.get("age_group","unknown")).lower().strip()
    if age_group not in ("infant","child","adolescent","adult","elderly"): age_group = "unknown"
    sex = str(pat.get("sex","unknown")).lower().strip()
    if sex not in ("male","female"): sex = "unknown"
    age_years = pat.get("age_years")
    try: age_years = int(age_years) if age_years else None
    except: age_years = None

    patient = {"age_group": age_group, "sex": sex}
    if age_years is not None: patient["age_years"] = age_years

    # Normalize onset, severity
    onset = parsed.get("onset_days")
    try: onset = int(onset) if onset else None
    except: onset = None
    severity = str(parsed.get("severity","unknown")).lower().strip()
    if severity not in ("mild","moderate","severe"): severity = "unknown"

    # Build encounter
    encounter = {
        "encounter_id": encounter_id,
        "location_id": location_id,
        "week_id": week_id,
        "note_text": note_text,
        "chw_id": str(parsed.get("chw_id", "unknown")),
        "visit_date": parsed.get("visit_date"),
        "visit_datetime": parsed.get("visit_datetime"),
        "encounter_sequence": parsed.get("encounter_sequence"),
        "area_id": str(parsed.get("area_id", "unknown")),
        "household_id": str(parsed.get("household_id", "unknown")),
        "gps": parsed.get("gps"),
        "patient": patient,
        "symptoms": symptoms,
        "other_symptoms": other_symp,
        "onset_days": onset,
        "severity": severity,
        "red_flags": parsed.get("red_flags", []),
        "treatments_given": [str(t) for t in parsed.get("treatments_given",[]) if t],
        "referral": None,
        "follow_up": None,
    }

    # Enforce evidence
    encounter, downgrades = enforce_evidence(encounter, note_text)
    result["evidence_downgrades"] = downgrades

    # Step 2: Syndrome tagging with MedGemma
    try:
        tag_p = syndrome_prompt.replace("{encounter_json}", json.dumps({
            "symptoms": encounter["symptoms"],
            "other_symptoms": encounter.get("other_symptoms", {}),
            "red_flags": encounter.get("red_flags", []),
            "severity": encounter.get("severity"),
            "onset_days": encounter.get("onset_days"),
        }, indent=2))
        tag_p = tag_p.replace("{note_text}", note_text)
        raw_tag = run_medgemma(tag_p)
        syndrome = parse_json_response(raw_tag) or {}
    except Exception as e:
        result["errors"].append(f"tagger_error: {e}")
        syndrome = {}

    tag = str(syndrome.get("syndrome_tag","unclear")).lower().strip()
    if tag not in ("respiratory_fever","acute_watery_diarrhea","other","unclear"):
        tag = "unclear"
    syndrome_result = {
        "encounter_id": encounter_id,
        "syndrome_tag": tag,
        "confidence": str(syndrome.get("confidence","low")).lower().strip(),
        "trigger_quotes": [str(q) for q in (syndrome.get("trigger_quotes") or []) if q][:5],
        "reasoning": str(syndrome.get("reasoning",""))[:300],
    }
    if not syndrome_result["trigger_quotes"]:
        syndrome_result["trigger_quotes"] = ["insufficient data"]

    # Step 3: Checklist with MedGemma
    try:
        cl_p = checklist_prompt.replace("{encounter_json}", json.dumps(encounter, indent=2, default=str))
        cl_p = cl_p.replace("{note_text}", note_text)
        raw_cl = run_medgemma(cl_p)
        checklist = parse_json_response(raw_cl) or {"questions": []}
    except Exception as e:
        result["errors"].append(f"checklist_error: {e}")
        checklist = {"questions": []}

    checklist_result = {
        "encounter_id": encounter_id,
        "questions": checklist.get("questions", [])[:5],
    }

    elapsed = time.time() - t0
    result["encounter"] = encounter
    result["syndrome_tag"] = syndrome_result
    result["checklist"] = checklist_result
    result["processing_time_s"] = round(elapsed, 2)

    return result

## 6. Run on Gold Notes

In [34]:
# Load gold notes
gold_path = ROOT / "data_synth" / "gold_encounters_merged.jsonl"
if not gold_path.exists():
    gold_path = ROOT / "data_synth" / "gold_encounters.jsonl"

gold_notes = [json.loads(l) for l in open(gold_path, encoding="utf-8")]
print(f"Loaded {len(gold_notes)} gold notes")

# For demo/time, run on a subset
N_DEMO = 20  # Change to len(gold_notes) for full run
demo_notes = gold_notes[:N_DEMO]
print(f"Running pipeline on {N_DEMO} notes...")

Loaded 443 gold notes
Running pipeline on 20 notes...


In [35]:
results = []
for i, note in enumerate(demo_notes):
    print(f"Processing {i+1}/{N_DEMO}: {note['encounter_id']}...", end=" ")
    r = process_note(
        note_text=note["note_text"],
        encounter_id=note["encounter_id"],
        location_id=note.get("location_id", "unknown"),
        week_id=note.get("week_id", 0),
    )
    print(f"→ {r['syndrome_tag']['syndrome_tag']} ({r['processing_time_s']}s)")
    if r["errors"]:
        print(f"  ⚠️ Errors: {r['errors']}")
    results.append(r)

print(f"\n✅ Processed {len(results)} notes")
avg_time = sum(r["processing_time_s"] for r in results) / len(results)
print(f"Average time per note: {avg_time:.1f}s")

Processing 1/20: gold_001... → respiratory_fever (71.01s)
Processing 2/20: gold_002... → respiratory_fever (67.67s)
Processing 3/20: gold_003... → respiratory_fever (72.94s)
Processing 4/20: gold_004... → respiratory_fever (64.75s)
Processing 5/20: gold_005... → respiratory_fever (90.24s)
Processing 6/20: gold_006... → respiratory_fever (76.18s)
Processing 7/20: gold_007... → respiratory_fever (83.06s)
Processing 8/20: gold_008... → respiratory_fever (64.73s)
Processing 9/20: gold_009... → respiratory_fever (76.4s)
Processing 10/20: gold_010... → respiratory_fever (71.45s)
Processing 11/20: gold_011... → respiratory_fever (74.2s)
Processing 12/20: gold_012... → respiratory_fever (72.08s)
Processing 13/20: gold_013... → respiratory_fever (75.96s)
Processing 14/20: gold_014... → respiratory_fever (77.23s)
Processing 15/20: gold_015... → other (80.83s)
Processing 16/20: gold_016... → respiratory_fever (80.21s)
Processing 17/20: gold_017... → respiratory_fever (76.4s)
Processing 18/20: gol

## 7. Evaluation

In [36]:
from collections import Counter

# Compare syndrome tags to gold labels
correct = 0
total = 0
confusion = {}

for r, gold in zip(results, demo_notes):
    predicted = r["syndrome_tag"]["syndrome_tag"]
    actual = gold.get("gold_syndrome_tag", "unclear")
    total += 1
    if predicted == actual:
        correct += 1
    key = (actual, predicted)
    confusion[key] = confusion.get(key, 0) + 1

accuracy = correct / total if total > 0 else 0
print(f"=== Syndrome Tag Accuracy ===")
print(f"Correct: {correct}/{total} = {accuracy:.1%}")
print()

# Confusion matrix
print("Confusion matrix (actual → predicted):")
actuals = sorted(set(k[0] for k in confusion))
preds = sorted(set(k[1] for k in confusion))
header = f"{'':>25}" + "".join(f"{p:>20}" for p in preds)
print(header)
for a in actuals:
    row = f"{a:>25}" + "".join(f"{confusion.get((a,p),0):>20}" for p in preds)
    print(row)

=== Syndrome Tag Accuracy ===
Correct: 19/20 = 95.0%

Confusion matrix (actual → predicted):
                                        other   respiratory_fever
        respiratory_fever                   1                  19


In [37]:
# Evidence quality
total_claims = 0
grounded_claims = 0
hallucinated_claims = 0
total_downgrades = 0

for r in results:
    enc = r["encounter"]
    note_lower = enc.get("note_text", "").lower()

    # Check symptoms
    for k, v in enc.get("symptoms", {}).items():
        if v.get("value") == "yes":
            total_claims += 1
            q = v.get("evidence_quote", "")
            if q and q.lower() in note_lower:
                grounded_claims += 1
            else:
                hallucinated_claims += 1

    # Check other_symptoms
    for k, v in enc.get("other_symptoms", {}).items():
        if v.get("value") == "yes":
            total_claims += 1
            q = v.get("evidence_quote", "")
            if q and q.lower() in note_lower:
                grounded_claims += 1
            else:
                hallucinated_claims += 1

    total_downgrades += len(r.get("evidence_downgrades", []))

print(f"=== Evidence Quality ===")
print(f"Total 'yes' claims: {total_claims}")
print(f"Grounded (quote in note): {grounded_claims}")
print(f"Hallucinated/ungrounded: {hallucinated_claims}")
if total_claims > 0:
    print(f"Grounding rate: {grounded_claims/total_claims:.1%}")
    print(f"Hallucination rate: {hallucinated_claims/total_claims:.1%}")
print(f"Evidence downgrades (post-enforcement): {total_downgrades}")

=== Evidence Quality ===
Total 'yes' claims: 0
Grounded (quote in note): 0
Hallucinated/ungrounded: 0
Evidence downgrades (post-enforcement): 111


In [38]:
# Per-symptom extraction stats
print(f"=== Per-Symptom Extraction ===")
symptom_stats = {}
for r in results:
    for k, v in r["encounter"]["symptoms"].items():
        if k not in symptom_stats:
            symptom_stats[k] = {"yes": 0, "no": 0, "unknown": 0}
        symptom_stats[k][v["value"]] += 1

for k in sorted(symptom_stats):
    s = symptom_stats[k]
    print(f"  {k:25s}: yes={s['yes']:3d}  no={s['no']:3d}  unknown={s['unknown']:3d}")

=== Per-Symptom Extraction ===
  bloody_diarrhea          : yes=  0  no=  0  unknown= 20
  cough                    : yes=  0  no=  0  unknown= 20
  difficulty_breathing     : yes=  0  no=  0  unknown= 20
  fever                    : yes=  0  no=  0  unknown= 20
  rash                     : yes=  0  no=  0  unknown= 20
  vomiting                 : yes=  0  no=  0  unknown= 20
  watery_diarrhea          : yes=  0  no=  0  unknown= 20


## 8. Surveillance — Anomaly Detection & SITREP

In [39]:
# Aggregate to weekly counts
records = []
for r in results:
    enc = r["encounter"]
    syn = r["syndrome_tag"]
    records.append({
        "week_id": enc.get("week_id", 0),
        "location_id": enc.get("location_id", "unknown"),
        "syndrome_tag": syn.get("syndrome_tag", "unclear"),
    })

df = pd.DataFrame(records)
weekly_counts = df.groupby(["week_id","location_id","syndrome_tag"]).size().reset_index(name="count")
print("Weekly counts:")
print(weekly_counts)

Weekly counts:
    week_id location_id       syndrome_tag  count
0         1       loc02  respiratory_fever      1
1         1       loc05  respiratory_fever      1
2         1       loc07  respiratory_fever      1
3         2       loc01  respiratory_fever      1
4         2       loc06  respiratory_fever      1
5         3       loc04  respiratory_fever      1
6         4       loc03  respiratory_fever      1
7         4       loc04  respiratory_fever      1
8         4       loc05  respiratory_fever      1
9         4       loc07  respiratory_fever      1
10        5       loc06              other      1
11        6       loc02  respiratory_fever      1
12        7       loc02  respiratory_fever      1
13        7       loc03  respiratory_fever      1
14        9       loc01  respiratory_fever      1
15        9       loc02  respiratory_fever      1
16        9       loc04  respiratory_fever      1
17       10       loc02  respiratory_fever      1
18       10       loc06  respirator

In [41]:
# --- Surveillance demo: load sim events -> aggregate -> anomaly detect ---

# Also load the full sim_events for surveillance demo
sim_path = ROOT / "data_synth" / "sim_events.csv"

if sim_path.exists():
    sim_df = pd.read_csv(sim_path)
    print(f"Loaded {len(sim_df)} simulation events for surveillance demo")
    print("Columns:", list(sim_df.columns))

    # Detect the syndrome column (handles 'syndrome_tag', 'syndrome', 'syndrome_*', etc.)
    syndrome_col = next((c for c in sim_df.columns if "syndrome" in c.lower()), None)
    if syndrome_col is None:
        raise ValueError(
            "No syndrome-like column found. Expected a column containing 'syndrome' in its name."
        )

    # Ensure week_id exists
    if "week_id" not in sim_df.columns:
        raise ValueError("sim_events.csv is missing required column: week_id")

    # Ensure location_id exists
    if "location_id" not in sim_df.columns:
        raise ValueError("sim_events.csv is missing required column: location_id")

    # Aggregate event-level rows into weekly counts expected by detect_anomalies()
    weekly_counts = (
        sim_df
        .groupby(["location_id", syndrome_col, "week_id"], dropna=False)
        .size()
        .reset_index(name="count")
        .rename(columns={syndrome_col: "syndrome_tag"})
    )

    # Make sure week_id is numeric (detect.py uses ranges/min/max)
    weekly_counts["week_id"] = pd.to_numeric(weekly_counts["week_id"], errors="raise").astype(int)

    print(f"Aggregated to {len(weekly_counts)} weekly counts rows")
    display(weekly_counts.head())

    # Import detector (make sure ROOT is on path if src lives there)
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))

    from src.detect import detect_anomalies

    anomalies = detect_anomalies(weekly_counts)

    print(f"\nAnomalies detected: {len(anomalies)}")
    if not anomalies.empty:
        print(anomalies.to_string(index=False))
else:
    print("No sim_events.csv found — skip surveillance demo")


Loaded 672 simulation events for surveillance demo
Columns: ['encounter_id', 'week_id', 'location_id', 'syndrome_tag', 'severity']
Aggregated to 225 weekly counts rows


,location_id,syndrome_tag,week_id,count
0,loc01,acute_watery_diarrhea,1,1
1,loc01,acute_watery_diarrhea,2,2
2,loc01,acute_watery_diarrhea,3,2
3,loc01,acute_watery_diarrhea,4,3
4,loc01,acute_watery_diarrhea,5,3



Anomalies detected: 8
 week_id location_id      syndrome_tag  count  baseline_mean  baseline_window_size
       2       loc02 respiratory_fever      5           2.00                     1
       2       loc03 respiratory_fever      5           2.00                     1
       7       loc04 respiratory_fever     15           4.75                     4
       8       loc04 respiratory_fever     17           7.25                     4
       7       loc05 respiratory_fever     17           3.00                     4
       8       loc05 respiratory_fever     19           6.75                     4
       1       loc06 respiratory_fever      5           0.00                     0
       1       loc07 respiratory_fever      5           0.00                     0


In [42]:
# Generate SITREP for the outbreak week using MedGemma
if sim_path.exists() and not anomalies.empty:
    outbreak_week = anomalies["week_id"].max()
    week_anomalies = anomalies[anomalies["week_id"] == outbreak_week]
    week_counts = sim_df[sim_df["week_id"] == outbreak_week]

    anomaly_csv = week_anomalies.to_csv(index=False)
    counts_csv = week_counts.to_csv(index=False)

    sitrep_p = sitrep_prompt.replace("{anomalies_csv}", anomaly_csv)
    sitrep_p = sitrep_p.replace("{weekly_counts_csv}", counts_csv)
    sitrep_p = sitrep_p.replace("{report_week}", str(outbreak_week))

    print(f"Generating SITREP for week {outbreak_week}...")
    t0 = time.time()
    raw_sitrep = run_medgemma(sitrep_p, max_new_tokens=1024)
    print(f"SITREP generated in {time.time()-t0:.1f}s")

    sitrep = parse_json_response(raw_sitrep)
    if sitrep:
        print(json.dumps(sitrep, indent=2))
    else:
        print("Raw output:")
        print(raw_sitrep[:800])

Generating SITREP for week 8...
SITREP generated in 31.9s
{
  "report_week": 8,
  "generated_at": "2024-03-08T00:00:00Z",
  "total_encounters": 250,
  "alerts": [
    {
      "location_id": "loc04",
      "syndrome_tag": "respiratory_fever",
      "current_count": 17,
      "baseline_mean": 7.25,
      "status": "alert"
    }
  ],
  "summary_text": "There is an increase in respiratory fever syndrome in location loc04. Respiratory fever is also elevated in locations loc01, loc02, loc03, loc05, loc06, loc07, and loc08. Acute watery diarrhea is present in location loc04.",
  "data_quality": {
    "completeness_pct": 1.0,
    "notes": "Syndromic classification only; no laboratory confirmation available."
  },
  "limitations": [
    "Syndromic classification only; no laboratory confirmation available."
  ]
}


## 9. Save Results

In [ ]:
# Save processed results
output = {
    "model": MEDGEMMA_ID,
    "n_notes_processed": len(results),
    "avg_processing_time_s": round(avg_time, 2),
    "syndrome_accuracy": round(accuracy, 3),
    "evidence_grounding_rate": round(grounded_claims / max(total_claims, 1), 3),
    "encounters": [r["encounter"] for r in results],
    "syndrome_tags": [r["syndrome_tag"] for r in results],
    "checklists": [r["checklist"] for r in results],
}

out_path = OUT_DIR / "pipeline_results.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, default=str)
print(f"Results saved to {out_path} ✅")

In [ ]:
print("=" * 60)
print("🎉 CHW Copilot Pipeline Complete!")
print("=" * 60)
print(f"Notes processed:        {len(results)}")
print(f"Syndrome accuracy:      {accuracy:.1%}")
print(f"Evidence grounding:     {grounded_claims}/{total_claims} ({grounded_claims/max(total_claims,1):.1%})")
print(f"Avg time per note:      {avg_time:.1f}s")
print(f"Model: {MEDGEMMA_ID}")